In [9]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
import time

spark = (SparkSession.builder
    .appName("FastFood-Partie3-DF")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.cores.max", "2")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate())

orders_schema = StructType([
    StructField("order_id", StringType()),
    StructField("restaurant_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("city", StringType()),
    StructField("status", StringType()),
    StructField("amount_mad", DoubleType()),
    StructField("items_count", IntegerType()),
    StructField("distance_km", DoubleType()),
    StructField("courier_id", StringType()),
    StructField("event_time", TimestampType()),
])
restaurants_schema = StructType([
    StructField("restaurant_id", StringType()),
    StructField("name", StringType()),
    StructField("city", StringType()),
    StructField("category", StringType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
])

orders = spark.read.schema(orders_schema).json("s3a://lake/bronze/history/orders_history.jsonl")
restaurants = (spark.read.schema(restaurants_schema).json("s3a://lake/reference/restaurants.json")
               .withColumnRenamed("city", "restaurant_city"))
orders.printSchema()
print(orders.count(), restaurants.count())

root
 |-- order_id: string (nullable = true)
 |-- restaurant_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount_mad: double (nullable = true)
 |-- items_count: integer (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- courier_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)

50000 200


In [8]:
clean = orders.dropDuplicates(["order_id", "status"]).cache()
delivered = clean.filter(F.col("status") == "DELIVERED").cache()
print("Événements nettoyés :", clean.count(), "| DELIVERED :", delivered.count())

t0 = time.time()
ca_df = delivered.groupBy("city").agg(F.round(F.sum("amount_mad"), 2).alias("ca_total_mad")).orderBy("city")
ca_df.show()
t_df = time.time() - t0

delivered.createOrReplaceTempView("delivered")
t0 = time.time()
spark.sql("""
    SELECT city, ROUND(SUM(amount_mad), 2) AS ca_total_mad
    FROM delivered GROUP BY city ORDER BY city
""").show()
t_sql = time.time() - t0
print(f"Temps DataFrame : {t_df:.2f} s ; SQL : {t_sql:.2f} s (RDD en Partie 2 : 0,39 s)")

26/11/13 23:20:25 WARN CacheManager: Asked to cache already cached data.
26/11/13 23:20:25 WARN CacheManager: Asked to cache already cached data.
                                                                                

Événements nettoyés : 48999 | DELIVERED : 15435


+----------+------------+
|      city|ca_total_mad|
+----------+------------+
|Casablanca|   694042.69|
| Marrakech|   704863.08|
|     Rabat|   596142.96|
|    Tanger|   583033.49|
+----------+------------+



[Stage 86:==================================================>   (186 + 3) / 200]

+----------+------------+
|      city|ca_total_mad|
+----------+------------+
|Casablanca|   694042.69|
| Marrakech|   704863.08|
|     Rabat|   596142.96|
|    Tanger|   583033.49|
+----------+------------+

Temps DataFrame : 2.63 s ; SQL : 2.77 s (RDD en Partie 2 : 0,39 s)


In [7]:
joined = delivered.join(restaurants.select("restaurant_id", "category"), "restaurant_id")
ca_cat = (joined.groupBy("city", "category")
                .agg(F.round(F.sum("amount_mad"), 2).alias("ca_mad"))
                .orderBy("city", F.desc("ca_mad")))
ca_cat.show(20)

[Stage 66:====================================================> (196 + 1) / 200]

+----------+--------+---------+
|      city|category|   ca_mad|
+----------+--------+---------+
|Casablanca| healthy|135342.79|
|Casablanca|   pizza|128977.83|
|Casablanca|marocain| 80301.26|
|Casablanca|   tacos| 77611.01|
|Casablanca| dessert| 77347.63|
|Casablanca|  burger| 73112.66|
|Casablanca|   sushi| 71418.63|
|Casablanca|  poulet| 49930.88|
| Marrakech| dessert|114702.67|
| Marrakech|   sushi|104808.67|
| Marrakech|marocain|102269.43|
| Marrakech|  poulet|101107.18|
| Marrakech|   tacos| 77956.78|
| Marrakech|   pizza| 77364.98|
| Marrakech|  burger| 76874.56|
| Marrakech| healthy| 49778.81|
|     Rabat|  burger|131388.12|
|     Rabat|marocain| 93428.77|
|     Rabat| healthy|  86963.8|
|     Rabat|   sushi| 79870.12|
+----------+--------+---------+
only showing top 20 rows



In [4]:
@F.udf(StringType())
def distance_band_udf(d):
    if d is None:
        return None
    if d < 2:
        return "court"
    if d <= 5:
        return "moyen"
    return "long"

d = F.col("distance_km")
distance_band_native = F.when(d < 2, "court").when(d <= 5, "moyen").otherwise("long")

t0 = time.time()
r1 = delivered.withColumn("band", distance_band_udf(d)).groupBy("band").count().collect()
t_udf = time.time() - t0

t0 = time.time()
r2 = delivered.withColumn("band", distance_band_native).groupBy("band").count().collect()
t_native = time.time() - t0

print(sorted(r1), sorted(r2))
print(f"UDF : {t_udf:.2f} s ; when/otherwise : {t_native:.2f} s")

[Stage 43:====================================================> (193 + 1) / 200]

[Row(band='court', count=2576), Row(band='long', count=7220), Row(band='moyen', count=5639)] [Row(band='court', count=2576), Row(band='long', count=7220), Row(band='moyen', count=5639)]
UDF : 9.65 s ; when/otherwise : 3.80 s


In [5]:
silver = (clean.join(restaurants.select("restaurant_id", "category"), "restaurant_id", "left")
               .withColumn("distance_band", distance_band_native))
silver.write.mode("overwrite").partitionBy("city").parquet("s3a://lake/silver/orders_history/")
print(spark.read.parquet("s3a://lake/silver/orders_history/").count())

[Stage 51:=====================================================>  (24 + 1) / 25]

48999


In [10]:
from pyspark.sql import functions as F

# Moins de partitions de shuffle : évite des centaines de petits fichiers
spark.conf.set("spark.sql.shuffle.partitions", "8")

# Bande de distance sans UDF (when/otherwise)
d = F.col("distance_km")
distance_band_native = F.when(d < 2, "court").when(d <= 5, "moyen").otherwise("long")

# Données nettoyées et enrichies
silver = (clean.join(restaurants.select("restaurant_id", "category"), "restaurant_id", "left")
               .withColumn("distance_band", distance_band_native))

# Une partition par ville, donc un seul fichier par dossier city=...
(silver.repartition("city")
       .write.mode("overwrite").partitionBy("city")
       .parquet("s3a://lake/silver/orders_history/"))

print(spark.read.parquet("s3a://lake/silver/orders_history/").count())

48999


In [13]:
spark.conf.set("spark.sql.shuffle.partitions", "8")

def timeit(f, n=3):
    ts = []
    for _ in range(n):
        t0 = time.time(); f(); ts.append(round(time.time() - t0, 2))
    return ts

print("DataFrame CA   :", timeit(lambda: delivered.groupBy("city").agg(F.sum("amount_mad")).collect()))
print("SQL CA         :", timeit(lambda: spark.sql("SELECT city, SUM(amount_mad) FROM delivered GROUP BY city").collect()))
print("UDF            :", timeit(lambda: delivered.withColumn("band", distance_band_udf(F.col("distance_km"))).groupBy("band").count().collect()))
print("when/otherwise :", timeit(lambda: delivered.withColumn("band", distance_band_native).groupBy("band").count().collect()))

DataFrame CA   : [0.74, 0.55, 0.53]
SQL CA         : [0.51, 0.45, 0.44]


UDF            : [1.84, 0.58, 0.56]
when/otherwise : [0.62, 0.34, 0.34]


In [12]:
spark.conf.set("spark.sql.shuffle.partitions", "8")

# Libérer l'ancien cache puis recalculer avec 8 partitions
clean.unpersist()
delivered.unpersist()
clean = orders.dropDuplicates(["order_id", "status"]).cache()
delivered = clean.filter(F.col("status") == "DELIVERED").cache()
delivered.createOrReplaceTempView("delivered")

print("Nettoyés :", clean.count(), "| DELIVERED :", delivered.count())
print("Partitions de delivered :", delivered.rdd.getNumPartitions())

[Stage 175:===================================>                     (5 + 2) / 8]

Nettoyés : 48999 | DELIVERED : 15435
Partitions de delivered : 8


In [14]:
q = (delivered.join(restaurants.select("restaurant_id", "category"), "restaurant_id")
              .groupBy("city", "category").agg(F.sum("amount_mad").alias("ca")))
q.explain(True)

== Parsed Logical Plan ==
'Aggregate ['city, 'category], ['city, 'category, sum('amount_mad) AS ca#12437]
+- Project [restaurant_id#4366, order_id#4365, customer_id#4367, city#4368, status#4369, amount_mad#4370, items_count#4371, distance_km#4372, courier_id#4373, event_time#4374, category#4388]
   +- Join Inner, (restaurant_id#4366 = restaurant_id#4385)
      :- Filter (status#4369 = DELIVERED)
      :  +- Deduplicate [order_id#4365, status#4369]
      :     +- Relation [order_id#4365,restaurant_id#4366,customer_id#4367,city#4368,status#4369,amount_mad#4370,items_count#4371,distance_km#4372,courier_id#4373,event_time#4374] json
      +- Project [restaurant_id#4385, category#4388]
         +- Project [restaurant_id#4385, name#4386, city#4387 AS restaurant_city#4397, category#4388, latitude#4389, longitude#4390]
            +- Relation [restaurant_id#4385,name#4386,city#4387,category#4388,latitude#4389,longitude#4390] json

== Analyzed Logical Plan ==
city: string, category: string, ca:

In [15]:
q.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (23)
+- HashAggregate (22)
   +- Exchange (21)
      +- HashAggregate (20)
         +- Project (19)
            +- BroadcastHashJoin Inner BuildRight (18)
               :- Filter (14)
               :  +- InMemoryTableScan (1)
               :        +- InMemoryRelation (2)
               :              +- AdaptiveSparkPlan (13)
               :                 +- Filter (12)
               :                    +- InMemoryTableScan (3)
               :                          +- InMemoryRelation (4)
               :                                +- AdaptiveSparkPlan (11)
               :                                   +- SortAggregate (10)
               :                                      +- Sort (9)
               :                                         +- Exchange (8)
               :                                            +- SortAggregate (7)
               :                                               +- Sort (6)
             